# LexRA — Legal Reasoning Assistant
### Fine-tuning TinyLlama on Indian Legal Dataset using LoRA

In [ ]:
# ── Step 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q transformers peft datasets accelerate bitsandbytes trl

In [ ]:
# ── Step 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/LexRA/model', exist_ok=True)
print('Drive mounted and folders ready')

In [ ]:
# ── Step 3: Upload Data Files ─────────────────────────────────────────────────
from google.colab import files

print('Upload train.jsonl')
files.upload()

print('Upload val.jsonl')
files.upload()

In [ ]:
# ── Step 4: Config ────────────────────────────────────────────────────────────
MODEL_NAME  = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
TRAIN_FILE  = 'train.jsonl'
VAL_FILE    = 'val.jsonl'
OUTPUT_DIR  = '/content/drive/MyDrive/LexRA/model'
MAX_LENGTH  = 512
EPOCHS      = 3
BATCH_SIZE  = 2
GRAD_ACCUM  = 8
LR          = 2e-4

In [ ]:
# ── Step 5: Load Data ─────────────────────────────────────────────────────────
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = Dataset.from_list(load_jsonl(TRAIN_FILE))
val_data   = Dataset.from_list(load_jsonl(VAL_FILE))

print(f'Train: {len(train_data)} | Val: {len(val_data)}')

In [ ]:
# ── Step 6: Tokenizer ─────────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

train_data = train_data.map(tokenize, batched=True, remove_columns=['text'])
val_data   = val_data.map(tokenize,   batched=True, remove_columns=['text'])
print('Tokenization done')

In [ ]:
# ── Step 7: Load Model with 4-bit Quantization ────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)
model = prepare_model_for_kbit_training(model)
print('Model loaded')

In [ ]:
# ── Step 8: Apply LoRA ────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Step 9: Train ─────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy='steps',
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print('Starting training...')
trainer.train()

In [ ]:
# ── Step 10: Save Model to Google Drive ───────────────────────────────────────
model.save_pretrained(OUTPUT_DIR + '/best_model')
tokenizer.save_pretrained(OUTPUT_DIR + '/best_model')
print('Model saved to Google Drive at:', OUTPUT_DIR + '/best_model')